# 🚀 Floor Plan Model Training — Google Drive Setup

### ✅ Pehle karo (ek baar):
1. Google Drive pe **`models/`** folder upload karo (poora folder — Dataset + Model_training.py sab)
2. Colab mein **Runtime → Change runtime type → T4 GPU** select karo
3. Phir neeche se ek ek cell run karo ▶️

In [ ]:
# ── STEP 1: GPU Check ──────────────────────────────────────────
!nvidia-smi
print("\n✅ GPU ready hai toh CUDA version dikhega upar")

In [ ]:
# ── STEP 2: Packages Install ───────────────────────────────────
!pip install -q torch torchvision opencv-python pillow numpy scipy scikit-learn tqdm
print("✅ Packages install ho gaye!")

In [ ]:
# ── STEP 3: Google Drive Mount ─────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
# Check karo ke models folder hai Drive mein
drive_models = '/content/drive/MyDrive/models'
if os.path.exists(drive_models):
    print("✅ models/ folder Drive mein mila!")
    print("Contents:", os.listdir(drive_models))
else:
    print("❌ ERROR: Drive mein models/ folder nahi mila!")
    print("👉 Pehle Google Drive pe models/ folder upload karo")

In [ ]:
# ── STEP 4: Drive se files Colab mein Copy ─────────────────────
import shutil, os

src = '/content/drive/MyDrive/models'
dst = '/content/models'

# Pehle se hai toh delete karo
if os.path.exists(dst):
    shutil.rmtree(dst)

print("📂 Drive se models/ copy ho raha hai... (1-2 min lagega)")
shutil.copytree(src, dst)

print("\n✅ Copy ho gaya!")
print("models/ contents:", os.listdir(dst))

# Dataset check karo
dataset_path = f'{dst}/Dataset'
if os.path.exists(dataset_path):
    print("\n📁 Dataset folders:")
    for folder in os.listdir(dataset_path):
        folder_path = f'{dataset_path}/{folder}'
        count = len(os.listdir(folder_path))
        print(f"   {folder}/ → {count} files")
else:
    print("❌ Dataset/ folder nahi mila! Drive mein check karo")

In [ ]:
# ── STEP 5: Output folder banao ────────────────────────────────
import os
os.makedirs('/content/models/model_output', exist_ok=True)

# Drive mein bhi output folder banao (save ke liye)
os.makedirs('/content/drive/MyDrive/models/model_output', exist_ok=True)

print("✅ Output folders ready!")

In [ ]:
# ── STEP 6: Training Start ─────────────────────────────────────
# Yeh cell 2-3 ghante chalega — band mat karna!
%cd /content
!python models/Model_training.py
print("\n🎉 Training complete!")

In [ ]:
# ── STEP 7: Trained Model Google Drive pe Save ─────────────────
import shutil, os, glob

output_drive = '/content/drive/MyDrive/models/model_output'
os.makedirs(output_drive, exist_ok=True)

# .pth files copy karo
for f in glob.glob('/content/models/model_output/*.pth'):
    shutil.copy(f, output_drive)
    print(f"✅ Saved: {os.path.basename(f)}")

# history JSON copy karo
for f in glob.glob('/content/models/model_output/*.json'):
    shutil.copy(f, output_drive)
    print(f"✅ Saved: {os.path.basename(f)}")

print("\n🎉 Model Drive pe save ho gaya!")
print(f"📁 Location: MyDrive/models/model_output/")

In [ ]:
# ── STEP 8: Training Results Graph ────────────────────────────
import json, matplotlib.pyplot as plt

history_path = '/content/models/model_output/retrain_history.json'

with open(history_path, 'r') as f:
    history = json.load(f)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Training Loss')

plt.subplot(1, 2, 2)
plt.plot(history['val_miou'], label='Val mIoU')
plt.xlabel('Epoch')
plt.ylabel('mIoU')
plt.legend()
plt.title('Validation mIoU')

plt.tight_layout()
plt.show()

print(f"🏆 Best mIoU: {max(history['val_miou']):.4f}")

In [ ]:
# ── STEP 9: Model Download (Optional) ─────────────────────────
# Drive se bhi le sakte ho, ya seedha download karo
from google.colab import files
import glob

for f in glob.glob('/content/models/model_output/*.pth'):
    print(f"Downloading: {f}")
    files.download(f)

print("✅ Download complete!")